In [ ]:
#Shubi's code to make all folds together file

import pandas as pd

from src.data_paths import (
    HUNTING_IS_CORRECT_ALL_FOLDS_PATH,
    HUNTING_IS_CORRECT_ITEMS_DIR,
    HUNTING_IS_CORRECT_SUBJECTS_DIR,
    IA_PARAGRAPH_PATH,
)

In [3]:
folds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# For Reread
# folds_folder = "../data/folds_RereadStratified"
# save_path = folds_folder + "/all_folds_subjects_items_RereadStratified.csv"
# subsplit_train = True

save_path = HUNTING_IS_CORRECT_ALL_FOLDS_PATH
subsplit_train = False

In [4]:
def get_fold_indices(
    i: int,
    num_folds: int = 10,
    use_double_test_size: bool = False,
    subsplit_train: bool = False,
) -> tuple[list[int], list[int], list[int]]:
    """
    Given a fold index i within the range [0, 9], return the indices for the test,
    validation, and training sets according to the specified folding strategy.

    Parameters:
    i (int): The fold index (should be between 0 and 9).

    Returns:
    tuple: A tuple containing the test indices (as a list), validation index (as an integer),
    and training indices (as a list).
    """
    if i < 0 or i > num_folds - 1:
        raise ValueError("Fold index must be within the range [0, 9].")

    validation_indices = [i]

    # modulo num_folds for the wraparound
    test_indices = [(i + 1) % num_folds]
    if use_double_test_size:
        test_indices.append((i + 2) % num_folds)

    # The rest are training indices
    train_indices = [
        x
        for x in range(num_folds)
        if x not in test_indices and x not in validation_indices
    ]

    if subsplit_train:
        test_indices = validation_indices + test_indices
        validation_indices = [
            v - 1 if v > 0 else num_folds - 1 for v in validation_indices
        ]
        train_indices = [
            x
            for x in range(num_folds)
            if x not in test_indices and x not in validation_indices
        ]

    print(
        f"Test folds: {test_indices}, \
            Validation fold: {validation_indices}, \
            Train folds: {train_indices}"
    )
    return test_indices, validation_indices, train_indices

In [5]:
all_subjects = []
all_items = []
for fold_index in folds:
    # open fold CSVs
    items = pd.read_csv(HUNTING_IS_CORRECT_ITEMS_DIR / f"fold_{fold_index}.csv", header=None)
    subjects = pd.read_csv(
        HUNTING_IS_CORRECT_SUBJECTS_DIR / f"fold_{fold_index}.csv", header=None
    )
    subjects.columns = ["subject"]
    items.columns = ["item"]
    subjects["fold"] = fold_index
    items["fold"] = fold_index
    all_subjects.append(subjects)
    all_items.append(items)

all_items = pd.concat(all_items).reset_index(drop=True)
all_subjects = pd.concat(all_subjects).reset_index(drop=True)

assert all_items.shape[0] == 30
assert all_subjects.shape[0] == 180
assert all_items.duplicated().sum() == 0
assert all_subjects.duplicated().sum() == 0


In [6]:
all_fold_dfs = []
for fold_idx in folds:
    # Get the fold indices
    test_indices, validation_indices, train_indices = get_fold_indices(
        fold_idx, subsplit_train=subsplit_train
    )

    all_train_subjects = all_subjects[all_subjects["fold"].isin(train_indices)].copy()
    all_train_items = all_items[all_items["fold"].isin(train_indices)].copy()

    # take the product of all_train_items and all_train_subjects
    all_train_subjects["key"] = 0
    all_train_items["key"] = 0
    all_train_df = all_train_subjects.merge(
        all_train_items, how="outer", on=["key"], suffixes=("_subjects", "_items")
    ).drop(columns=["key"])
    all_train_df["eval_regime"] = "train"
    all_train_df["eval_type"] = "train"

    all_df = all_train_df.to_dict(orient="records")
    for eval_type, eval_indices in [
        ("test", test_indices),
        ("val", validation_indices),
    ]:
        items = []
        subjects = []
        for fold_index in eval_indices:
            # open fold CSVs
            items.append(all_items.loc[all_items["fold"] == fold_index, "item"].values)
            subjects.append(
                all_subjects.loc[all_subjects["fold"] == fold_index, "subject"].values
            )

        items = set([item for sublist in items for item in sublist])
        subjects = set([subject for sublist in subjects for subject in sublist])

        for participant_id in all_subjects["subject"].unique():
            for article_id in all_items["item"].unique():
                is_subject_in_fold = participant_id in subjects
                is_item_in_fold = article_id in items
                is_subject_in_train = (
                    participant_id in all_train_subjects["subject"].values
                )
                is_item_in_train = article_id in all_train_items["item"].values

                if eval_type == "val":
                    if is_subject_in_fold and is_item_in_fold:
                        regime = "new_item_and_subject"
                    elif is_subject_in_train and is_item_in_fold:
                        regime = "new_item"
                    elif is_subject_in_fold and is_item_in_train:
                        regime = "new_subject"
                    else:
                        continue

                elif eval_type == "test":
                    if is_subject_in_fold and is_item_in_fold:
                        regime = "new_item_and_subject"
                    elif not is_subject_in_fold and is_item_in_fold:
                        regime = "new_item"
                    elif is_subject_in_fold and not is_item_in_fold:
                        regime = "new_subject"
                    else:
                        continue
                else:
                    raise ValueError("Invalid eval_type")

                all_df.append(
                    {
                        "subject": participant_id,
                        "item": article_id,
                        "eval_regime": regime,
                        "eval_type": eval_type,
                    }
                )

    all_df = pd.DataFrame(all_df).drop(columns=["fold_subjects", "fold_items"])
    all_df["fold"] = fold_idx
    all_fold_dfs.append(all_df)

all_df = pd.concat(all_fold_dfs).reset_index(drop=True)

Test folds: [1],             Validation fold: [0],             Train folds: [2, 3, 4, 5, 6, 7, 8, 9]
Test folds: [2],             Validation fold: [1],             Train folds: [0, 3, 4, 5, 6, 7, 8, 9]
Test folds: [3],             Validation fold: [2],             Train folds: [0, 1, 4, 5, 6, 7, 8, 9]
Test folds: [4],             Validation fold: [3],             Train folds: [0, 1, 2, 5, 6, 7, 8, 9]
Test folds: [5],             Validation fold: [4],             Train folds: [0, 1, 2, 3, 6, 7, 8, 9]
Test folds: [6],             Validation fold: [5],             Train folds: [0, 1, 2, 3, 4, 7, 8, 9]
Test folds: [7],             Validation fold: [6],             Train folds: [0, 1, 2, 3, 4, 5, 8, 9]
Test folds: [8],             Validation fold: [7],             Train folds: [0, 1, 2, 3, 4, 5, 6, 9]
Test folds: [9],             Validation fold: [8],             Train folds: [0, 1, 2, 3, 4, 5, 6, 7]
Test folds: [0],             Validation fold: [9],             Train folds: [1, 2, 3, 4, 5,

In [7]:
# ia_report = pd.read_csv(
#     "../ln_shared_data/onestop/OneStop_v1_20250126/lacclab_processed/ia_Paragraph.csv",
#     engine="pyarrow",
#     usecols=["participant_id", "article_batch", "article_id"],
# ).drop_duplicates()
ia_report = pd.read_csv(
    IA_PARAGRAPH_PATH,
    engine="pyarrow",
    usecols=["participant_id", "article_batch", "article_id"],
).drop_duplicates()
ia_report["item"] = (
    ia_report["article_batch"].astype(str) + "_" + ia_report["article_id"].astype(str)
)

In [8]:
# keep only the combination of items and subjects that are in the IA report
all_df_real = all_df.merge(
    ia_report,
    how="inner",
    left_on=["subject", "item"],
    right_on=["participant_id", "item"],
).drop(columns=["item", "subject"])
all_df_real.to_csv(save_path, index=False)

In [9]:
all_df_real[all_df_real["fold"] == 0]

,eval_regime,eval_type,fold,participant_id,article_batch,article_id
0,train,train,0,l10_39,1,9
1,train,train,0,l10_39,1,2
2,train,train,0,l10_39,1,3
3,train,train,0,l10_39,1,5
4,train,train,0,l10_39,1,7
...,...,...,...,...,...,...
1795,new_item,val,0,l53_166,1,1
1796,new_item,val,0,l55_435,3,6
1797,new_item,val,0,l58_189,1,1
1798,new_item,val,0,l59_547,3,6


In [10]:
# a = pd.read_csv(
#     "../ln_shared_data/onestop/OneStop_v1_20250126/lacclab_processed/ia_Paragraph.csv",
#     engine="pyarrow",
#     usecols=["participant_id", "article_batch", "article_id"],
# )
a = pd.read_csv(
    IA_PARAGRAPH_PATH,
    engine="pyarrow",
    usecols=["participant_id", "article_batch", "article_id"],
)

In [11]:
a.merge(
    all_df_real[all_df_real["fold"] == 0],
    how="inner",
    on=["participant_id", "article_batch", "article_id"],
)

,participant_id,article_batch,article_id,eval_regime,eval_type,fold
0,l42_2070,3,6,new_item,val,0
1,l42_2070,3,6,new_item,val,0
2,l42_2070,3,6,new_item,val,0
3,l42_2070,3,6,new_item,val,0
4,l42_2070,3,6,new_item,val,0
...,...,...,...,...,...,...
1266510,l10_39,1,10,new_item,test,0
1266511,l10_39,1,10,new_item,test,0
1266512,l10_39,1,10,new_item,test,0
1266513,l10_39,1,10,new_item,test,0


In [12]:
z = (
    all_df_real.value_counts(["eval_type", "eval_regime"])
    .reset_index()
    .sort_values(by="count", ascending=False)
    .drop_duplicates(subset=["eval_regime", "eval_type"])
)
z

,eval_type,eval_regime,count
0,train,train,11520
1,test,new_item,1620
2,test,new_subject,1620
3,val,new_subject,1440
4,val,new_item,1440
5,test,new_item_and_subject,180
6,val,new_item_and_subject,180


In [16]:
subjects_count = (
    all_df_real.groupby(["eval_regime", "eval_type", "fold"])["participant_id"]
    .nunique()
    .reset_index()
    .sort_values(by="participant_id", ascending=False)
)
subjects_count.columns = ["eval_regime", "eval_type", "fold", "num_subjects"]
subjects_count

,eval_regime,eval_type,fold,num_subjects
0,new_item,test,0,162
1,new_item,test,1,162
2,new_item,test,2,162
3,new_item,test,3,162
4,new_item,test,4,162
...,...,...,...,...
59,new_subject,val,9,18
55,new_subject,val,5,18
54,new_subject,val,4,18
56,new_subject,val,6,18


In [6]:
import pandas as pd
from src.data_paths import HUNTERS_FOLDS_DIR, GATHERERS_FOLDS_DIR

hunters_fold0 = pd.read_csv(HUNTERS_FOLDS_DIR / "fold_0_trial_ids_by_regime.csv")
gatherers_fold0 = pd.read_csv(GATHERERS_FOLDS_DIR / "fold_0_trial_ids_by_regime.csv")

# regime mapping: new_item -> *_seen_subject_unseen_item, new_item_new_participant -> *_unseen_subject_unseen_item
regime_map = {
    "new_item": ["test_seen_subject_unseen_item", "val_seen_subject_unseen_item"],
    "new_item_new_participant": ["test_unseen_subject_unseen_item", "val_unseen_subject_unseen_item"],
}

for label, regimes in regime_map.items():
    h_texts = set(hunters_fold0.loc[hunters_fold0["regime"].isin(regimes), "unique_paragraph_id"].unique())
    g_texts = set(gatherers_fold0.loc[gatherers_fold0["regime"].isin(regimes), "unique_paragraph_id"].unique())

    print(f"=== {label} ===")
    print(f"  hunters: {len(h_texts)} unique texts")
    print(f"  gatherers: {len(g_texts)} unique texts")
    print(f"  identical sets: {h_texts == g_texts}")
    print(f"  intersection: {len(h_texts & g_texts)}")
    print(f"  only in hunters: {sorted(h_texts - g_texts)}")
    print(f"  only in gatherers: {sorted(g_texts - h_texts)}")
    print()

=== new_item ===
  hunters: 80 unique texts
  gatherers: 80 unique texts
  identical sets: False
  intersection: 54
  only in hunters: ['2_2_Adv_1', '2_2_Adv_2', '2_2_Adv_3', '2_2_Adv_4', '2_2_Adv_5', '2_2_Adv_6', '2_2_Ele_1', '2_2_Ele_2', '2_2_Ele_3', '2_2_Ele_4', '2_2_Ele_5', '2_2_Ele_6', '3_6_Adv_1', '3_6_Adv_2', '3_6_Adv_3', '3_6_Adv_4', '3_6_Adv_5', '3_6_Adv_6', '3_6_Adv_7', '3_6_Ele_1', '3_6_Ele_2', '3_6_Ele_3', '3_6_Ele_4', '3_6_Ele_5', '3_6_Ele_6', '3_6_Ele_7']
  only in gatherers: ['2_4_Adv_1', '2_4_Adv_2', '2_4_Adv_3', '2_4_Adv_4', '2_4_Adv_5', '2_4_Adv_6', '2_4_Adv_7', '2_4_Ele_1', '2_4_Ele_2', '2_4_Ele_3', '2_4_Ele_4', '2_4_Ele_5', '2_4_Ele_6', '2_4_Ele_7', '3_10_Adv_1', '3_10_Adv_2', '3_10_Adv_3', '3_10_Adv_4', '3_10_Adv_5', '3_10_Adv_6', '3_10_Ele_1', '3_10_Ele_2', '3_10_Ele_3', '3_10_Ele_4', '3_10_Ele_5', '3_10_Ele_6']

=== new_item_new_participant ===
  hunters: 80 unique texts
  gatherers: 80 unique texts
  identical sets: False
  intersection: 54
  only in hunters: ['

In [11]:
# Compare hunters fold 0 vs every gatherer fold
regime_map = {
    "new_item": ["test_seen_subject_unseen_item", "val_seen_subject_unseen_item"],
    "new_item_new_participant": ["test_unseen_subject_unseen_item", "val_unseen_subject_unseen_item"],
}

h0 = pd.read_csv(HUNTERS_FOLDS_DIR / "fold_0_trial_ids_by_regime.csv")
h_texts_by_regime = {
    label: set(h0.loc[h0["regime"].isin(regimes), "unique_paragraph_id"].unique())
    for label, regimes in regime_map.items()
}

cross = []
for g_fold in range(10):
    g = pd.read_csv(GATHERERS_FOLDS_DIR / f"fold_{g_fold}_trial_ids_by_regime.csv")
    for label, regimes in regime_map.items():
        g_texts = set(g.loc[g["regime"].isin(regimes), "unique_paragraph_id"].unique())
        h_texts = h_texts_by_regime[label]
        cross.append({
            "hunter_fold": 0,
            "gatherer_fold": g_fold,
            "regime": label,
            "n_hunters": len(h_texts),
            "n_gatherers": len(g_texts),
            "identical": h_texts == g_texts,
            "intersection": len(h_texts & g_texts),
            "only_hunters": len(h_texts - g_texts),
            "only_gatherers": len(g_texts - h_texts),
        })

cross_df = pd.DataFrame(cross)
cross_df

,hunter_fold,gatherer_fold,regime,n_hunters,n_gatherers,identical,intersection,only_hunters,only_gatherers
0,0,0,new_item,80,80,False,54,26,26
1,0,0,new_item_new_participant,80,80,False,54,26,26
2,0,1,new_item,80,76,False,26,54,50
3,0,1,new_item_new_participant,80,76,False,26,54,50
4,0,2,new_item,80,66,False,12,68,54
5,0,2,new_item_new_participant,80,66,False,12,68,54
6,0,3,new_item,80,60,False,0,80,60
7,0,3,new_item_new_participant,80,60,False,0,80,60
8,0,4,new_item,80,64,False,0,80,64
9,0,4,new_item_new_participant,80,64,False,0,80,64


In [12]:
# Build "GatherersRefolded": gatherer participant split + hunter item split (per fold).
# Trials whose (subject_group, item_group) do not form a coherent regime
# (val_subject x test_item, or test_subject x val_item) are dropped.
from pathlib import Path

GATHERERS_REFOLDED_DIR = GATHERERS_FOLDS_DIR.parent / "GatherersRefolded"
GATHERERS_REFOLDED_DIR.mkdir(exist_ok=True)


def split_subjects(fold_df):
    train_mask = (fold_df["regime"] == "train_train") | fold_df["regime"].str.contains("_seen_subject_")
    val_mask = fold_df["regime"].str.startswith("val_unseen_subject")
    test_mask = fold_df["regime"].str.startswith("test_unseen_subject")
    return (
        set(fold_df.loc[train_mask, "participant_id"].unique()),
        set(fold_df.loc[val_mask, "participant_id"].unique()),
        set(fold_df.loc[test_mask, "participant_id"].unique()),
    )


def split_items(fold_df):
    train_mask = (fold_df["regime"] == "train_train") | fold_df["regime"].str.endswith("_seen_item")
    val_mask = fold_df["regime"].str.startswith("val_") & fold_df["regime"].str.endswith("_unseen_item")
    test_mask = fold_df["regime"].str.startswith("test_") & fold_df["regime"].str.endswith("_unseen_item")
    return (
        set(fold_df.loc[train_mask, "unique_paragraph_id"].unique()),
        set(fold_df.loc[val_mask, "unique_paragraph_id"].unique()),
        set(fold_df.loc[test_mask, "unique_paragraph_id"].unique()),
    )


regime_lookup = {
    ("train", "train"): "train_train",
    ("train", "val"): "val_seen_subject_unseen_item",
    ("train", "test"): "test_seen_subject_unseen_item",
    ("val", "train"): "val_unseen_subject_seen_item",
    ("val", "val"): "val_unseen_subject_unseen_item",
    ("test", "train"): "test_unseen_subject_seen_item",
    ("test", "test"): "test_unseen_subject_unseen_item",
}

summary = []
for fold_idx in range(10):
    g = pd.read_csv(GATHERERS_FOLDS_DIR / f"fold_{fold_idx}_trial_ids_by_regime.csv")
    h = pd.read_csv(HUNTERS_FOLDS_DIR / f"fold_{fold_idx}_trial_ids_by_regime.csv")

    g_train_s, g_val_s, g_test_s = split_subjects(g)
    h_train_i, h_val_i, h_test_i = split_items(h)

    subj_group_map = (
        {s: "train" for s in g_train_s}
        | {s: "val" for s in g_val_s}
        | {s: "test" for s in g_test_s}
    )
    item_group_map = (
        {it: "train" for it in h_train_i}
        | {it: "val" for it in h_val_i}
        | {it: "test" for it in h_test_i}
    )

    out = g.copy()
    out["subj_group"] = out["participant_id"].map(subj_group_map)
    out["item_group"] = out["unique_paragraph_id"].map(item_group_map)
    out["new_regime"] = [
        regime_lookup.get((s, i)) for s, i in zip(out["subj_group"], out["item_group"])
    ]

    n_total = len(out)
    n_dropped = out["new_regime"].isna().sum()
    out = out[out["new_regime"].notna()].copy()
    out["regime"] = out["new_regime"]
    out = out.drop(columns=["subj_group", "item_group", "new_regime"])

    out.to_csv(GATHERERS_REFOLDED_DIR / f"fold_{fold_idx}_trial_ids_by_regime.csv", index=False)

    summary.append({
        "fold": fold_idx,
        "n_total_input": n_total,
        "n_kept": len(out),
        "n_dropped": int(n_dropped),
        **out["regime"].value_counts().to_dict(),
    })

summary_df = pd.DataFrame(summary).fillna(0)
print(f"Wrote 10 fold files to: {GATHERERS_REFOLDED_DIR}")
summary_df

Wrote 10 fold files to: E:\QA PROJECT\programming\QA_eyetracking_workspace\data\GatherersRefolded


,fold,n_total_input,n_kept,n_dropped,train_train,val_seen_subject_unseen_item,test_seen_subject_unseen_item,val_unseen_subject_seen_item,test_unseen_subject_seen_item,val_unseen_subject_unseen_item,test_unseen_subject_unseen_item
0,0,9718,9478,240,5856,960,959,732,731,120,120
1,1,9718,9490,228,5952,959,864,743,744,120,108
2,2,9718,9508,210,6094,864,816,762,762,108,102
3,3,9718,9526,192,6238,816,720,780,780,102,90
4,4,9718,9550,168,6430,720,624,804,804,90,78
5,5,9718,9562,156,6527,624,624,816,815,78,78
6,6,9718,9562,156,6528,624,623,815,816,78,78
7,7,9718,9532,186,6287,623,864,786,786,78,108
8,8,9718,9520,198,6190,864,720,774,774,108,90
9,9,9718,9508,210,6094,720,960,762,762,90,120


In [13]:
# Verify: in each refolded fold, every "unseen_subject" is truly absent from
# all training rows, and every "unseen_item" is truly absent from all training rows.
TRAIN_REGIMES = {
    "train_train",
    "val_seen_subject_unseen_item",
    "test_seen_subject_unseen_item",
    "val_unseen_subject_seen_item",
    "test_unseen_subject_seen_item",
}
# Of the above, which expose a TRAINING subject vs a TRAINING item:
TRAIN_SUBJECT_REGIMES = {
    "train_train",
    "val_seen_subject_unseen_item",
    "test_seen_subject_unseen_item",
}
TRAIN_ITEM_REGIMES = {
    "train_train",
    "val_unseen_subject_seen_item",
    "test_unseen_subject_seen_item",
}
UNSEEN_SUBJECT_REGIMES = {
    "val_unseen_subject_seen_item",
    "val_unseen_subject_unseen_item",
    "test_unseen_subject_seen_item",
    "test_unseen_subject_unseen_item",
}
UNSEEN_ITEM_REGIMES = {
    "val_seen_subject_unseen_item",
    "val_unseen_subject_unseen_item",
    "test_seen_subject_unseen_item",
    "test_unseen_subject_unseen_item",
}

issues = []
for fold_idx in range(10):
    df = pd.read_csv(GATHERERS_REFOLDED_DIR / f"fold_{fold_idx}_trial_ids_by_regime.csv")

    train_subjects = set(df.loc[df["regime"].isin(TRAIN_SUBJECT_REGIMES), "participant_id"].unique())
    train_items = set(df.loc[df["regime"].isin(TRAIN_ITEM_REGIMES), "unique_paragraph_id"].unique())

    unseen_subjects = set(df.loc[df["regime"].isin(UNSEEN_SUBJECT_REGIMES), "participant_id"].unique())
    unseen_items = set(df.loc[df["regime"].isin(UNSEEN_ITEM_REGIMES), "unique_paragraph_id"].unique())

    bad_subjects = unseen_subjects & train_subjects
    bad_items = unseen_items & train_items

    issues.append({
        "fold": fold_idx,
        "n_train_subjects": len(train_subjects),
        "n_unseen_subjects": len(unseen_subjects),
        "subjects_violating": len(bad_subjects),
        "n_train_items": len(train_items),
        "n_unseen_items": len(unseen_items),
        "items_violating": len(bad_items),
        "bad_subjects_sample": sorted(bad_subjects)[:5],
        "bad_items_sample": sorted(bad_items)[:5],
    })

issues_df = pd.DataFrame(issues)
print("Subject leakage violations total:", issues_df["subjects_violating"].sum())
print("Item leakage violations total:   ", issues_df["items_violating"].sum())
issues_df

Subject leakage violations total: 0
Item leakage violations total:    0


,fold,n_train_subjects,n_unseen_subjects,subjects_violating,n_train_items,n_unseen_items,items_violating,bad_subjects_sample,bad_items_sample
0,0,144,36,0,244,80,0,[],[]
1,1,144,36,0,248,76,0,[],[]
2,2,144,36,0,254,70,0,[],[]
3,3,144,36,0,260,64,0,[],[]
4,4,144,36,0,268,56,0,[],[]
5,5,144,36,0,272,52,0,[],[]
6,6,144,36,0,272,52,0,[],[]
7,7,144,36,0,262,62,0,[],[]
8,8,144,36,0,258,66,0,[],[]
9,9,144,36,0,254,70,0,[],[]
